# <font color="#418FDE" size="6.5" uppercase>**B: Machine Learning in PyTorch**</font>
----

> Last update: 20240722

By the end of this lecture, you will be able to:

* Develop simple machine learning models in PyTorch.


## **1. Machine Learning with PyTorch in General**

Machine learning involves the use of algorithms & atistical models to enable computers to perform tasks without explicit instructions. [PyTorch](https://pytorch.org/) is an open-source machine learning library primarily developed by [Facebook’s AI Research lab (FAIR)](https://ai.meta.com/research/). It’s widely used for deep learning applications & is known for its dynamic computation graph & simplicity. Here’s a brief introduction to getting started with PyTorch:

Key Concepts in PyTorch:

* **Tensors:** The primary data structure in PyTorch, similar to NumPy arrays but with the added advantage of being able to run on GPUs.
* **Autograd:** PyTorch’s automatic differentiation library that helps in building & training neural networks.
* **Modules:** The building blocks for neural networks in PyTorch, similar to Keras in TensorFlow.
* **Optimizers:** Methods to update the weights of the network based on the gradients.

Basic Workflow in PyTorch:

* **Data Preparation:** Load & preprocess data using torchvision for images, torchtext for text, etc.
* **Model Definition:** Define a neural network by subclassing nn.Module.
* **Loss Function:** Define a loss function (e.g., nn.CrossEntropyLoss).
* **Optimizer:** Choose an optimizer (e.g., torch.optim.SGD).
* **Training Loop:** Write a loop to train the model, compute the loss, backpropagate, & update the weights.

## **2. Neural Networks with PyTorch**

In this section, we will see some simple examples of a neural network in PyTorch:

In [ ]:
#@title Linear Regression with PyTorch
'''
Runtime: CPU
torch.nn:          https://pytorch.org/docs/stable/nn.html
torch.nn.Linear    https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
'''
# Importing necessary libraries
import torch  # PyTorch library
import torch.nn as nn  # Neural network module
import matplotlib.pyplot as plt  # Plotting library

# Generating Synthetic Data
# -------------------------

# Create a tensor X with values ranging from 1 to 50, reshaped to a column vector
X = torch.linspace(1, 50, 50).reshape(-1, 1)

# Set the random seed for reproducibility
torch.manual_seed(71)

# Generate random noise tensor e with values from -8 to 8, same shape as X
e = torch.randint(-8, 9, (50, 1), dtype=torch.float32)

# Define the target variable y as a linear function of X with added noise
y = 2 * X + 1 + e

# Plot the synthetic data
plt.figure(figsize=[4, 4])
plt.scatter(X.numpy(), y.numpy())
plt.plot([0, 50], [0, 100], 'r-', linewidth=2)

# Defining a Simple Linear Model
# ------------------------------

# Set the random seed for reproducibility
torch.manual_seed(59)

# Define a custom linear model class inheriting from nn.Module
# This class will represent our linear regression model
class LinearModel(nn.Module):
    def __init__(self, in_features, out_features):
        # Initialize the parent class (nn.Module) to ensure proper setup
        super(LinearModel, self).__init__()
        # Define a linear layer with given input & output features
        # nn.Linear applies a linear transformation to the incoming data: y = x*W^T + b
        # https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        # Define the forward pass for the model
        # Pass the input tensor x through the linear layer
        # The linear layer computes the output (prediction) y_pred
        y_pred = self.linear(x)
        # Return the computed prediction
        return y_pred

# Plotting the Initial Model Prediction (before training)
# -------------------------------------------------------

# Instantiate the custom model
model = LinearModel(1, 1)

# Print initial weights & bias
# model.linear.weight accesses the weight parameter of the linear layer in the model
# .item() converts the weight tensor to a Python scalar
print(f'Initial weight: {model.linear.weight.item()}')

# model.linear.bias accesses the bias parameter of the linear layer in the model
# .item() converts the bias tensor to a Python scalar
print(f'Initial bias: {model.linear.bias.item()}')

# Generate input tensor for plotting
x1 = torch.linspace(0.0, 50.0, 50).view(50, 1) # easy reshaping with .view!

# Get model predictions for the input tensor
y1 = model.forward(x1)

# Plot the synthetic data & initial model prediction

# Create a new figure with a specified size of 4x4 inches
plt.figure(figsize=[4, 4])

# Plot the synthetic data points as a scatter plot
# X.numpy() & y.numpy() convert the PyTorch tensors to NumPy arrays for plotting
plt.scatter(X.numpy(), y.numpy())

# Plot the initial model prediction as a line plot
# x1.numpy() converts the input tensor x1 to a NumPy array
# y1.detach().numpy() converts the prediction tensor y1 to a NumPy array after detaching it from the computation graph
# 'r' specifies the color red for the line
# linewidth=2 sets the width of the line to 2 points
plt.plot(x1.numpy(), y1.detach().numpy(), 'r', linewidth=2)

# Show the plot
plt.show()

# Training the Model
# ------------------

# Define the loss function (Mean Squared Error)
# nn.MSELoss computes the mean squared error (MSE) between the predicted values & the actual values
# This loss function is suitable for regression problems
criterion = nn.MSELoss()

# Define the optimizer (Stochastic Gradient Descent) with learning rate 0.001
# torch.optim.SGD is an optimizer that implements stochastic gradient descent
# model.parameters() returns an iterator over the model parameters (weights & biases)
# lr=0.001 sets the learning rate for the optimizer, controlling the step size during gradient descent
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# Set the number of epochs for training
epochs = 50

# Initialize a list to store loss values
losses = []

# Training loop
for epoch in range(epochs):
    # Forward pass: compute predicted y by passing input data X to the model
    # The forward method of the model calculates the predicted output y_pred
    y_pred = model.forward(X)

    # Compute the loss: measure the discrepancy between the predicted y & the true y
    # criterion calculates the Mean Squared Error (MSE) between y_pred & y
    loss = criterion(y_pred, y)

    # Append the current loss to the list of losses for later analysis
    # .item() converts the loss tensor to a Python number
    losses.append(loss.item())

    # Print the current epoch number & the corresponding loss
    # Epoch number is formatted to be three digits with leading zeros
    # Loss is formatted to six decimal places
    print(f'Epoch: {epoch+1:03d}/{epochs} | Loss: {loss.item():.6f}')

    # Zero gradients: clear old gradients from the last step
    # This is necessary because gradients by default add up; they need to be reset
    optimizer.zero_grad()

    # Backward pass: compute gradient of the loss with respect to model parameters
    # This is done via backpropagation
    loss.backward()

    # Update weights: adjust model parameters based on the gradients
    # optimizer.step() applies the gradient descent step, updating the parameters
    optimizer.step()

# Print trained weights & bias
# model.linear.weight accesses the weight parameter of the linear layer in the model
# .item() converts the weight tensor to a Python scalar
print(f'Trained weight: {model.linear.weight.item()}')

# model.linear.bias accesses the bias parameter of the linear layer in the model
# .item() converts the bias tensor to a Python scalar
print(f'Trained bias: {model.linear.bias.item()}')

# Plotting the Final Model Prediction

# Get model predictions after training
y1 = model.forward(x1)

# Plot the synthetic data & final model prediction
plt.figure(figsize=[4, 4])
plt.scatter(X.numpy(), y.numpy())
plt.plot(x1.numpy(), y1.detach().numpy(), 'r-', linewidth=2)

# Plotting the Loss Over Epochs

# Plot the loss over epochs
plt.figure(figsize=[4, 4])
plt.plot(range(1, epochs + 1), losses, 'b-', linewidth=2)
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')

plt.show()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

In [ ]:
#@title Simple Classification with PyTorch
'''
Runtime: CPU
tensors:               https://pytorch.org/docs/stable/tensors.html
torch.nn:              https://pytorch.org/docs/stable/nn.html
torch.nn.Linear        https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
torch.nn.functional:   https://pytorch.org/docs/stable/nn.functional.html
torch.utils.data:      https://pytorch.org/docs/stable/data.html
torch.nn.Module:       https://pytorch.org/docs/stable/generated/torch.nn.Module.html
torch.no_grad():       https://pytorch.org/docs/stable/generated/torch.no_grad.html
'''

# PyTorch library
import torch

# Neural network module
import torch.nn as nn

# Plotting library
import matplotlib.pyplot as plt

# torch.nn.functional contains functions that are useful when defining the forward pass of a neural network
# These functions are stateless & operate on input data
# https://pytorch.org/docs/stable/nn.functional.html
import torch.nn.functional as F

# TensorDataset is a dataset wrapping tensors, useful for combining input & target tensors
# DataLoader is an iterator that provides data in batches, useful for training & testing the model
# https://pytorch.org/docs/stable/data.html
from torch.utils.data import TensorDataset, DataLoader

# pandas is a powerful data manipulation library, often used for data preprocessing & analysis
import pandas as pd

# train_test_split is a utility function to split datasets into training & testing subsets
# Useful for evaluating the performance of a machine learning model
from sklearn.model_selection import train_test_split

# Load the Iris Dataset

# Read the Iris dataset from a CSV file into a pandas DataFrame
# The CSV file is located at the specified URL
# pandas.read_csv reads the CSV file & stores the data in a DataFrame
df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')

# Extract the feature columns (all columns except the last one) & convert to a NumPy array
# .iloc[:, :-1] selects all rows & all columns except the last one
# .values converts the DataFrame to a NumPy array
# torch.FloatTensor converts the NumPy array to a PyTorch tensor of type float
#  https://pytorch.org/docs/stable/tensors.html
X = torch.FloatTensor(df.iloc[:, :-1].values)

# Extract the target column (the last column) & convert to categorical integer labels
# pd.factorize converts the species names to integer labels
# [0] selects the first element of the tuple returned by pd.factorize, which is the array of labels
# torch.LongTensor converts the array of labels to a PyTorch tensor of type long (integer)
# https://pytorch.org/docs/stable/tensors.html
y = torch.LongTensor(pd.factorize(df['species'])[0])

# Split the dataset into training & testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

# Using DataLoader for Batch Processing

# Create a TensorDataset from the feature & target tensors
# TensorDataset wraps the feature tensor X & target tensor y into a dataset
# This dataset can be used with DataLoader to iterate over batches
dataset = TensorDataset(X, y)

# Create a DataLoader from the dataset
# DataLoader provides an iterable over the dataset with support for batching, shuffling, & parallel data loading
# batch_size=10 specifies that each batch should contain 10 samples
# shuffle=True indicates that the data should be shuffled at each epoch
loader = DataLoader(dataset, batch_size=10, shuffle=True)

# Iterate over the DataLoader for batch processing

# Enumerate over the DataLoader to get both the batch index (b) & the batch data (batch)
# DataLoader returns batches of data & their corresponding labels
for b, batch in enumerate(loader):
    # Print the batch index & the batch data
    # batch contains a tuple (features, labels) for the current batch
    print(b, batch)

    # Break the loop after the first iteration to only process & print the first batch
    # This is just an example
    break

# Defining a Neural Network Model for Classification

# Define a custom neural network model class inheriting from nn.Module
# This class will represent a neural network for classification tasks
class ClassificationModel(nn.Module):
    def __init__(self, input_size, output_size, hidden_layers):
        # Initialize the parent class (nn.Module) to ensure proper setup
        # https://pytorch.org/docs/stable/generated/torch.nn.Module.html
        super(ClassificationModel, self).__init__()

        # Initialize an empty list to store the layers of the network
        layers = []

        # Iterate over the hidden_layers list to create hidden layers
        for i in range(len(hidden_layers)):
            # Add a linear layer
            # The input size of the first layer is input_size
            # For subsequent layers, the input size is the output size of the previous layer
            layers.append(nn.Linear(input_size if i == 0 else hidden_layers[i-1], hidden_layers[i]))

            # Add a ReLU activation function after each linear layer
            layers.append(nn.ReLU())

        # Add the final linear layer connecting the last hidden layer to the output layer
        layers.append(nn.Linear(hidden_layers[-1], output_size))

        # Use nn.Sequential to create a sequential container of layers
        # This container will pass the input through each layer in the order they were added
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # Define the forward pass for the model
        # Pass the input tensor x through the sequential container of layers
        return self.network(x)

# Create an instance of the ClassificationModel class
# input_size=4: The number of input features (e.g., 4 features in the Iris dataset)
# output_size=3: The number of output classes (e.g., 3 species of Iris)
# hidden_layers=[8, 9]: A list specifying the number of neurons in each hidden layer
# (two hidden layers with 8 & 9 neurons respectively)
model = ClassificationModel(input_size=4, output_size=3, hidden_layers=[8, 9])

# Define the loss function (Cross Entropy Loss)
# nn.CrossEntropyLoss combines nn.LogSoftmax & nn.NLLLoss in one single class
# It is useful for multi-class classification problems
criterion = nn.CrossEntropyLoss()

# Define the optimizer (Adam) with learning rate 0.01
# torch.optim.Adam is an optimizer that implements the Adam algorithm
# model.parameters() returns an iterator over the model parameters (weights & biases)
# lr=0.01 sets the learning rate for the optimizer, controlling the step size during gradient descent
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Set the number of epochs for training
# epochs specifies how many times the entire dataset will be passed through the model
epochs = 100

# Initialize a list to store loss values for each epoch
losses = []

# Initialize a list to store accuracy values for each epoch
accuracies = []

# Define a function to compute metrics (loss & accuracy) for the model

def metrics(model, inputs, outputs):
    # Disable gradient calculation (no_grad context) as we are in evaluation mode
    # https://pytorch.org/docs/stable/generated/torch.no_grad.html
    with torch.no_grad():
        # Compute the model's predictions for the given inputs
        y_pred = model(inputs)

        # Compute the loss using the criterion (Cross Entropy Loss)
        # .item() converts the loss tensor to a Python scalar
        loss = criterion(y_pred, outputs).item()

        # Compute the accuracy
        # y_pred.argmax(dim=1) gives the predicted class labels (indices of the max log-probability)
        # == outputs compares the predicted labels with the true labels
        # .float() converts the boolean tensor to a float tensor (True becomes 1.0 & False becomes 0.0)
        # .mean() computes the mean accuracy
        # .item() converts the accuracy tensor to a Python scalar
        # "dim" here is similar to axis in numpy & TensorFlow
        accuracy = (y_pred.argmax(dim=1) == outputs).float().mean().item()

        # Return the computed loss & accuracy
        return loss, accuracy

# Training the Classification Model

# Iterate over the number of epochs
for epoch in range(epochs):
    # Iterate over the DataLoader to get batches of data
    for batch in loader:
        # Unpack the batch into features (X_batch) & labels (y_batch)
        X_batch, y_batch = batch

        # Zero the gradients to prevent accumulation from previous steps
        optimizer.zero_grad()

        # Forward pass: compute predicted y by passing X_batch to the model
        y_pred = model(X_batch)

        # Compute the loss using the criterion (Cross Entropy Loss)
        loss = criterion(y_pred, y_batch)

        # Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()

        # Update weights: adjust model parameters based on the gradients
        optimizer.step()

    # After processing all batches in the current epoch, compute the training metrics
    train_loss, train_accuracy = metrics(model, X, y)

    # Append the computed loss & accuracy to their respective lists for later analysis
    losses.append(train_loss)
    accuracies.append(train_accuracy)

    # Print the loss & accuracy every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {train_loss:.5f} | Accuracy: {train_accuracy:.4f}')

# Plotting Training Loss & Accuracy

# Create a new figure with a specified size of 8x4 inches
plt.figure(figsize=[8, 4])

# Create a subplot for the training loss
plt.subplot(1, 2, 1)
# Plot the training loss over epochs
# range(epochs) provides the x-axis values (epoch numbers)
# losses provides the y-axis values (loss values)
plt.plot(range(epochs), losses, label='Loss')
# Set the x-axis label to 'Epochs'
plt.xlabel('Epochs')
# Set the y-axis label to 'Loss'
plt.ylabel('Loss')
# Display the legend to label the plot
plt.legend()

# Create a subplot for the training accuracy
plt.subplot(1, 2, 2)
# Plot the training accuracy over epochs
# range(epochs) provides the x-axis values (epoch numbers)
# accuracies provides the y-axis values (accuracy values)
# color='orange' sets the color of the accuracy plot to orange
plt.plot(range(epochs), accuracies, label='Accuracy', color='orange')
# Set the x-axis label to 'Epochs'
plt.xlabel('Epochs')
# Set the y-axis label to 'Accuracy'
plt.ylabel('Accuracy')
# Display the legend to label the plot
plt.legend()

# Adjust the layout to ensure subplots do not overlap
plt.tight_layout()

# Show the plots
plt.show()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **3. MNIST Classification with PyTorch & GPU**

This section will show an example of developing & training a Convolutional Neural Network using PyTorch for the MNIST dataset.

In [ ]:
#@title Import necessary libraries
'''
Runtime: GPU $$$
tensors:                    https://pytorch.org/docs/stable/tensors.html
torch.nn:                   https://pytorch.org/docs/stable/nn.html
torch.nn.Linear             https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
torch.nn.functional:        https://pytorch.org/docs/stable/nn.functional.html
torch.utils.data:           https://pytorch.org/docs/stable/data.html
torch.nn.Module:            https://pytorch.org/docs/stable/generated/torch.nn.Module.html
torch.no_grad():            https://pytorch.org/docs/stable/generated/torch.no_grad.html
torch.optim:                https://pytorch.org/docs/stable/optim.html
torchvision:                https://pytorch.org/vision/stable/index.html
torchvision.transforms:     https://pytorch.org/vision/stable/transforms.html
torchvision.datasets:       https://pytorch.org/vision/main/datasets.html
'''

# Import necessary libraries

# Import the PyTorch library
# PyTorch is an open-source machine learning library used for applications such as computer vision & natural language processing
import torch

# Import the neural network module from PyTorch
# torch.nn provides classes & functions to build neural networks
import torch.nn as nn

# Import the optimization module from PyTorch
# torch.optim provides various optimization algorithms such as SGD, Adam, etc.
import torch.optim as optim

# Import the torchvision library
# torchvision provides tools to handle image & video data
import torchvision

# Import the transforms module from torchvision
# torchvision.transforms provides common image transformations for data augmentation & preprocessing
import torchvision.transforms as transforms

# Import the matplotlib library for plotting
# matplotlib.pyplot provides functions to create various types of plots & visualizations
import matplotlib.pyplot as plt

# Import the tqdm library for progress bars
# tqdm provides a fast, extensible progress bar for loops & tasks
from tqdm import tqdm

# Import the numpy library
# numpy provides support for large, multi-dimensional arrays & matrices, along with a collection of mathematical functions to operate on these arrays
import numpy as np


In [ ]:
#@title Convolutional neural network
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # First convolutional layer: 1 input channel (grayscale), 32 output channels, 5x5 kernel
        self.layer1 = nn.Sequential(
            #https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            #https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html
            nn.BatchNorm2d(32),  # Batch normalization for faster convergence
            # https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html
            nn.ReLU(),  # Activation function
            # https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
            nn.MaxPool2d(kernel_size=2, stride=2))  # Max pooling layer

        # Second convolutional layer: 32 input channels, 64 output channels, 5x5 kernel
        self.layer2 = nn.Sequential(
            #https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            #https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html
            nn.BatchNorm2d(64),  # Batch normalization
            # https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html
            nn.ReLU(),  # Activation function
            # https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
            nn.MaxPool2d(kernel_size=2, stride=2))  # Max pooling layer

        # Fully connected layer: input size 7*7*64, output size 1000
        self.fc1 = nn.Linear(7*7*64, 1000)
        # Fully connected layer: input size 1000, output size 10 (number of classes)
        self.fc2 = nn.Linear(1000, 10)

    def forward(self, x):
        # Forward pass through the first convolutional layer
        out = self.layer1(x)
        # Forward pass through the second convolutional layer
        out = self.layer2(out)
        # Flatten the output for the fully connected layer
        out = out.reshape(out.size(0), -1) # also look at https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html
        # Forward pass through the first fully connected layer
        out = self.fc1(out)
        # Forward pass through the second fully connected layer
        out = self.fc2(out)
        return out


In [ ]:
#@title Track accuracy
def track_accuracy(model, loader):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device)  # Move images to the configured device
            labels = labels.to(device)  # Move labels to the configured device
            outputs = model(images)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted class
            total += labels.size(0)  # Increment total by the number of labels
            correct += (predicted == labels).sum().item()  # Increment correct by the number of correct predictions
        accuracy = 100 * correct / total  # Calculate accuracy
    model.train()  # Set the model back to training mode
    return accuracy


In [ ]:
#@title Training
def training(model, num_epochs, loader, criterion, optimizer):
    # Train the model
    total_step = len(train_loader)  # Total number of batches
    loss_list = []  # List to store loss values
    acc_train = []  # List to store training accuracy values
    acc_test = []  # List to store testing accuracy values

    # Loop over the number of epochs
    for epoch in range(num_epochs):
        # Loop over each batch in the training DataLoader

        pbar = tqdm(train_loader,
                    desc = f"Training: {epoch + 1:03d}/{num_epochs}",
                    ncols = 125,
                    leave = True)


        # Creating running loss & accuracy empty lists
        running_loss = []
        running_accuracy = []

        for i, (images, labels) in enumerate(pbar, 1):
            # Important note: In PyTorch, the images in a batch is typically represented as (batch_size, channels, height, width)
            # For example, (100, 1, 28, 28) for the MNIST data
            images = images.to(device)  # Move images to the configured device
            labels = labels.to(device)  # Move labels to the configured device

            # Forward pass: compute predicted y by passing x to the model
            outputs = model(images)
            # Compute the loss
            loss = criterion(outputs, labels)
            running_loss.append(loss.item())

            # Backward pass & optimize
            optimizer.zero_grad()  # Zero the gradients
            loss.backward()  # Backward pass
            optimizer.step()  # Update weights

            # Add the runnign loss
            pbar.set_postfix(loss = f"{sum(running_loss)/i : 10.6f}")

        # Get the average of all losses in the running_loss as the loss of the current epoch
        loss_list.append(sum(running_loss)/i)

        # Track accuracy on the train set
        acc_train.append(track_accuracy(model, train_loader))

        # Track accuracy on the test set
        acc_test.append(track_accuracy(model, test_loader))

    return model, loss_list, acc_train, acc_test


In [ ]:
#@title Plot
def plot(loss_list, acc_train, acc_test):
    # Plot the loss & accuracy curves
    plt.figure(figsize=(10, 4))

    # Plot the training loss over iterations
    plt.subplot(1, 2, 1)
    plt.plot(loss_list)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title(f"Training Loss")

    # Plot the train & test accuracies over epochs
    plt.subplot(1, 2, 2)
    plt.plot(acc_train)
    plt.plot(acc_test)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f"Model Accuracy")
    plt.legend(['Training', 'Testing'])

    plt.show()


In [ ]:
#@title Main run
# Device configuration (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
num_epochs = 10  # Number of times the entire dataset is passed through the model
batch_size = 1000  # Number of samples per batch to be passed through the model
learning_rate = 0.001  # Step size for parameter updates

# MNIST dataset
# Download & load the training dataset, applying a transformation to convert images to PyTorch tensors
train_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)

# Download & load the test dataset, applying a transformation to convert images to PyTorch tensors
test_dataset = torchvision.datasets.MNIST(root='./data',
                                          train=False,
                                          transform=transforms.ToTensor())

# Data loader
# DataLoader provides an iterable over the dataset with support for batching, shuffling, & parallel data loading
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

# DataLoader for the test dataset, used for evaluating the model
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

# Create an instance of the model & move it to the configured device (GPU/CPU)
model = ConvNet().to(device)

# Print the model for observation
# You can use the following libraries to observe the flowchart of the models similar to TensorFlow
# from torchview import draw_graph; from torchviz import make_dot
# see more at:
# https://github.com/mert-kurttutan/torchview
# https://github.com/szagoruyko/pytorchviz
print(model)

# Loss & optimizer
# CrossEntropyLoss combines nn.LogSoftmax & nn.NLLLoss in one single class
criterion = nn.CrossEntropyLoss()
# Adam optimizer with the specified learning rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training the model
model, loss_list, acc_train, acc_test = training(model, num_epochs,
                                                 train_loader, criterion, optimizer)
# Make some plots!
plot(loss_list, acc_train, acc_test)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


## **4. Transfer Learning & Fine-Tuning with PyTorch**

This section will show an example of transfer learning & fine-tuning an ImageNet pre-trained MobileNet Version 2 mode using PyTorch for the CIFAR10 dataset.

In [ ]:
#@title Necessary Libraries
'''
Runtime: GPU $$$
tensors:                    https://pytorch.org/docs/stable/tensors.html
torch.nn:                   https://pytorch.org/docs/stable/nn.html
torch.nn.Linear             https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
torch.nn.functional:        https://pytorch.org/docs/stable/nn.functional.html
torch.utils.data:           https://pytorch.org/docs/stable/data.html
torch.nn.Module:            https://pytorch.org/docs/stable/generated/torch.nn.Module.html
torch.no_grad():            https://pytorch.org/docs/stable/generated/torch.no_grad.html
torch.optim:                https://pytorch.org/docs/stable/optim.html
torchvision:                https://pytorch.org/vision/stable/index.html
torchvision.transforms:     https://pytorch.org/vision/stable/transforms.html
torchvision.datasets:       https://pytorch.org/vision/main/datasets.html
torchvision.models:         https://pytorch.org/vision/stable/models.html
'''
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
#@title Tracking Accuracy
def track_accuracy(model, loader):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device)  # Move images to the configured device
            labels = labels.to(device)  # Move labels to the configured device
            outputs = model(images)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted class
            total += labels.size(0)  # Increment total by the number of labels
            correct += (predicted == labels).sum().item()  # Increment correct by the number of correct predictions
        accuracy = 100 * correct / total  # Calculate accuracy
    model.train()  # Set the model back to training mode
    return accuracy


In [ ]:
#@title Training
def training(model, num_epochs, loader, criterion, optimizer, mode = 'Transfer_Learning'):
    # Train the model
    total_step = len(train_loader)  # Total number of batches
    loss_list = []  # List to store loss values
    acc_train = []  # List to store training accuracy values
    acc_test = []  # List to store testing accuracy values

    # Loop over the number of epochs
    for epoch in range(num_epochs):
        # Loop over each batch in the training DataLoader

        pbar = tqdm(loader,
                    desc = f"{mode}: {epoch + 1:03d}/{num_epochs}",
                    ncols = 125,
                    leave = True)

        # Creating running loss & accuracy empty lists
        running_loss = []
        running_accuracy = []

        for i, (images, labels) in enumerate(pbar, 1):
            # Important note: In PyTorch, the images in a batch is typically represented as (batch_size, channels, height, width)
            # For example, (100, 1, 28, 28) for the MNIST data
            images = images.to(device)  # Move images to the configured device
            labels = labels.to(device)  # Move labels to the configured device

            # Forward pass: compute predicted y by passing x to the model
            outputs = model(images)
            # Compute the loss
            loss = criterion(outputs, labels)
            running_loss.append(loss.item())

            # Backward pass & optimize
            optimizer.zero_grad()  # Zero the gradients
            loss.backward()  # Backward pass
            optimizer.step()  # Update weights

            # Add the runnign loss
            pbar.set_postfix(loss = f"{sum(running_loss)/i : 10.6f}")

        # Get the average of all losses in the running_loss as the loss of the current epoch
        loss_list.append(sum(running_loss)/i)

        # Track accuracy on the train set
        acc_train.append(track_accuracy(model, train_loader))

        # Track accuracy on the test set
        acc_test.append(track_accuracy(model, test_loader))

    return model, loss_list, acc_train, acc_test


In [ ]:
#@title Plot
def plot(loss_list, acc_train, acc_test, mode = 'Transfer_Learning'):
    # Plot the loss & accuracy curves
    plt.figure(figsize=(10, 4))

    # Plot the training loss over iterations
    plt.subplot(1, 2, 1)
    plt.plot(loss_list)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title(f"{mode} | Training Loss")

    # Plot the train & test accuracies over epochs
    plt.subplot(1, 2, 2)
    plt.plot(acc_train)
    plt.plot(acc_test)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f"{mode} | Model Accuracy")
    plt.legend(['Training', 'Testing'])

    plt.show()


In [ ]:
#@title Main Run
# Device configuration (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
num_epochs_transfer_learning = 5  # Number of times the entire dataset is passed through the model in transfer learning
num_epochs_fine_tuning = 5 # Number of times the entire dataset is passed through the model in fine tuning
batch_size = 128  # Number of samples per batch to be passed through the model
learning_rate_transfer_learning = 0.01  # Step size for parameter updates
learning_rate_fine_tuning = 0.00001  # Step size for parameter updates

# Data augmentation & normalization for training
# Just normalization for validation
# The values used in transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) are the mean & standard deviation
# of the CIFAR-10 dataset. These statistics are precomputed from the entire CIFAR-10 training set & are used to normalize the
# images.
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # Randomly crop the image with padding
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.ToTensor(),  # Convert the image to PyTorch tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),  # Normalize the image
])

transform_test = transforms.Compose([
    transforms.ToTensor(),  # Convert the image to PyTorch tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),  # Normalize the image
])

# CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(root='./data',
                                             train=True,
                                             transform=transform_train,
                                             download=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data',
                                            train=False,
                                            transform=transform_test)

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

# Load the pre-trained MobileNetV2 model
model = mobilenet_v2(pretrained=True)

# Freeze all the layers except the last one
for param in model.parameters():
    param.requires_grad = False

# Modify the final layer to match the number of classes in CIFAR-10
# The original MobileNetV2 is trained on ImageNet with 1000 classes
model.classifier = nn.Sequential(nn.Linear(model.last_channel, 512),
                                 nn.ReLU(),
                                 nn.Linear(512, 256),
                                 nn.ReLU(),
                                 nn.Linear(256, 10))

# Print the model
# print(model)

# Only the parameters of the last layer should be trainable
params_to_update = model.classifier.parameters()

# Move the model to the configured device (GPU/CPU)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss combines nn.LogSoftmax & nn.NLLLoss in one single class

# Adam optimizer with the specified learning rate
optimizer = optim.Adam(params_to_update, lr=learning_rate_transfer_learning)

# Transfer learning
model, loss_list, acc_train, acc_test  = training(model,
                                                  num_epochs_transfer_learning,
                                                  train_loader, criterion, optimizer, mode = 'Transfer_Learning')

# Make some plots!
plot(loss_list, acc_train, acc_test, mode = 'Transfer_Learning')

# Unfreeze all the layers
for param in model.parameters():
    param.requires_grad = True

# All parameters should be trainable
params_to_update = model.parameters()

# Loss & optimizer
criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss combines nn.LogSoftmax & nn.NLLLoss in one single class
optimizer = optim.Adam(params_to_update, lr=learning_rate_fine_tuning)  # Adam optimizer with the specified learning rate

# Fine tuning
model, loss_list, acc_train, acc_test  = training(model,
                                                  num_epochs_transfer_learning,
                                                  train_loader, criterion, optimizer, mode = 'Fine_tuning')

# Make some plots!
plot(loss_list, acc_train, acc_test, mode = 'Fine_tuning')

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


# <font color="#418FDE" size="6.5" uppercase>**B: Machine Learning in PyTorch**</font>
----

In this lecture, you learned to:

* Develop simple machine learning models in PyTorch.

In the next Module (Module 8), we will go over "Visual Studio Code, Git, & GitHub".